# Week 06 - Webtoon Scraping Pattern

| 항목 | 내용 |
|---|---|
| 이름 | 이성민 |
| 학과 | 소프트웨어융합과 |
| 학년 | 2학년 |
| 학번 | 2151050 |
| 시트 기준 열 | 0410 webtoon |

이 노트북은 과제 요구사항을 학습용으로 재구성한 설명형 산출물이다. 원본 코드를 그대로 복사하지 않고, 같은 개념을 작은 로컬 예제로 다시 구현한다.

## 목표

동적 페이지 크롤링이 필요한 이유를 이해하고 웹툰 목록 파싱 로직을 재현한다.

모든 코드는 외부 서비스 접속 없이 실행되도록 구성했다. 실제 API나 웹사이트를 사용할 때는 같은 처리 흐름에서 데이터 입력 부분만 교체하면 된다.

## 1. 동적 페이지 크롤링의 핵심

네이버 웹툰처럼 JavaScript로 내용이 늦게 그려지는 페이지는 브라우저 렌더링 후 HTML을 읽어야 한다. 실행 검증을 위해 여기서는 렌더링이 끝난 HTML 조각을 샘플로 사용한다.

In [1]:
from html.parser import HTMLParser
import pandas as pd

rendered_html = '''
<ul id="webtoon-list">
  <li class="toon"><a href="/toon/1">데이터 탐정</a><span class="author">Lee</span><span class="score">9.81</span></li>
  <li class="toon"><a href="/toon/2">API 고양이</a><span class="author">Kim</span><span class="score">9.65</span></li>
  <li class="toon"><a href="/toon/3">크롤링 연구소</a><span class="author">Park</span><span class="score">9.72</span></li>
</ul>
'''

class WebtoonParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.rows = []
        self.current = None
        self.field = None

    def handle_starttag(self, tag, attrs):
        attrs = dict(attrs)
        if tag == "li" and attrs.get("class") == "toon":
            self.current = {}
        elif self.current is not None and tag == "a":
            self.field = "title"
            self.current["url"] = attrs.get("href", "")
        elif self.current is not None and tag == "span":
            self.field = attrs.get("class")

    def handle_data(self, data):
        text = data.strip()
        if self.current is not None and self.field and text:
            self.current[self.field] = text

    def handle_endtag(self, tag):
        if tag == "li" and self.current is not None:
            self.rows.append(self.current)
            self.current = None
        self.field = None

parser = WebtoonParser()
parser.feed(rendered_html)
webtoons = pd.DataFrame(parser.rows)
webtoons["score"] = webtoons["score"].astype(float)
webtoons

,url,title,author,score
0,/toon/1,데이터 탐정,Lee,9.81
1,/toon/2,API 고양이,Kim,9.65
2,/toon/3,크롤링 연구소,Park,9.72


## 2. 대기 조건을 함수로 모델링

Selenium의 `WebDriverWait`는 특정 요소가 나타날 때까지 기다린다. 여기서는 브라우저를 띄우지 않고도 같은 사고방식을 함수로 표현한다.

In [2]:
def wait_until_rows_exist(rows, minimum=1):
    if len(rows) < minimum:
        raise ValueError("렌더링된 웹툰 목록이 아직 충분하지 않습니다.")
    return rows

checked = wait_until_rows_exist(webtoons, minimum=3)
top_webtoon = checked.sort_values("score", ascending=False).iloc[0].to_dict()

assert top_webtoon["title"] == "데이터 탐정"
assert checked["url"].str.startswith("/toon/").all()
top_webtoon

{'url': '/toon/1', 'title': '데이터 탐정', 'author': 'Lee', 'score': 9.81}

## 3. 결과 저장 형태

크롤링 과제의 결과물은 보통 CSV, DataFrame, 또는 간단한 리포트다. 데이터가 정리되어 있으면 이후 시각화나 서비스 연동도 쉬워진다.

In [3]:
report = {
    "count": len(webtoons),
    "average_score": round(webtoons["score"].mean(), 2),
    "best_title": top_webtoon["title"],
}
csv_preview = webtoons.to_csv(index=False)

assert {"title", "author", "score", "url"}.issubset(webtoons.columns)
report

{'count': 3, 'average_score': np.float64(9.73), 'best_title': '데이터 탐정'}